In [ ]:
# This imports the OpenContextAPI from the api.py file in the
# opencontext directory.
%run '../opencontext/api.py'

In [ ]:
import numpy as np
import pandas as pd

oc_api = OpenContextAPI()

# Clear old cached records.
oc_api.clear_api_cache()

# This is a search url for bovid tibias.
url = 'https://opencontext.org/query/?cat=oc-gen-cat-bio-subj-ecofact---oc-gen-cat-animal-bone&prop=obo-foodon-00001303---gbif-1---gbif-44---gbif-359---gbif-731---gbif-9614&prop=oc-zoo-has-anat-id---obo-uberon-0002091---obo-uberon-0000979&type=subjects#tab=0/aq=facet-cidoc-crm-p2-has-type---obo-uberon-0000979/ovgrd=oc/zm=3/lat=38.82/lng=-22.50/ov=sqr'
# Fetch the 'standard' (linked data identified) attributes in use with
# data at the url.
stnd_attribs_tuples = oc_api.get_standard_attributes(
    url,
    # The optional argument below gets popular standard
    # zooarchaeological (bone) measurements.
    add_von_den_driesch_bone_measures=True
)

# Now display the standard attributes found in this search / query result
for slug, label in stnd_attribs_tuples:
    print(f'{label}, identified by slug: {slug}')

In [ ]:
# Make a list of only the slugs from the list of slug, label tuples.
stnd_attribs = [slug for slug, _ in stnd_attribs_tuples]

# Make a dataframe by fetching result records from Open Context.
# This will be slow until we finish improvements to Open Context's API.
# However, the results get cached by saving as files locally. That
# makes iterating on this notebook much less painful.
df = oc_api.url_to_dataframe(url, stnd_attribs)

In [ ]:
# Display a sample of the dataframe.
df.head(10)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Now we're going to make a plot.
markers = [
    'o',
    'x',
    'v',
    'D',
    'p',
    '^',
    's',
    '*',
]
colors = cm.rainbow(np.linspace(0, 1, len(markers)))
taxa = df['Has taxonomic identifier'].unique().tolist()
ax = None
i = 0
for taxon in df['Has taxonomic identifier'].unique().tolist():
    taxon_index = (
        (df['Has taxonomic identifier'] == taxon)
        & ~df['Bp'].isnull()
        & ~df['Dp'].isnull()
    )
    if df[taxon_index].empty:
        # No data for this taxon
        continue
    label = '{} [n={}]'.format(taxon, len(df[taxon_index].index))
    if not ax:
        ax = df[taxon_index].plot.scatter(
            x='Bp', 
            y='Dp', 
            marker=markers[i], 
            label=label, 
            color=colors[i].reshape(1,-1)
        )
    else:
        df[taxon_index].plot.scatter(
            x='Bp', 
            y='Dp', 
            marker=markers[i], 
            label=label, 
            ax=ax, 
            color=colors[i].reshape(1,-1)
        )
    i += 1
    if i >= len(markers):
        # This is here to make sure we don't run out of markers.
        markers += [(m + m) for m in markers]
        colors = cm.rainbow(np.linspace(0, 1, len(markers)))





In [ ]:
import os

# Now save the results of all of this as a CSV file.
repo_path = os.path.dirname(os.path.abspath(os.getcwd()))
csv_path = os.path.join(
    repo_path, 
    'files',
    'oc-api-demo-results.csv'
)
df.to_csv(csv_path, index=False)
print('Saved this example as a CSV table at: {}'.format(csv_path))
